In [1]:
import numpy as np
import tensorflow as tf

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

In [2]:
data = load_iris()

X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", data.target_names)

X shape: (150, 4)
y shape: (150,)
Classes: ['setosa' 'versicolor' 'virginica']


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Standardize input features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (120, 4)
Testing shape: (30, 4)


In [6]:
encoder = OneHotEncoder(sparse_output=False)

y_train_encoded = encoder.fit_transform(
    y_train.reshape(-1, 1)
)

y_test_encoded = encoder.transform(
    y_test.reshape(-1, 1)
)

print(y_train[:5])
print(y_train_encoded[:5])

[0 2 1 0 1]
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]]


In [7]:
model = Sequential([
    Input(shape=(4,)),

    Dense(16, activation="relu"),
    Dense(8, activation="relu"),

    Dense(3, activation="softmax")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
history = model.fit(
    X_train,
    y_train_encoded,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 0.4792 - loss: 1.0259 - val_accuracy: 0.6667 - val_loss: 0.9476
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.5208 - loss: 0.9992 - val_accuracy: 0.6667 - val_loss: 0.9206
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5729 - loss: 0.9690 - val_accuracy: 0.6667 - val_loss: 0.8939
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.6250 - loss: 0.9420 - val_accuracy: 0.7500 - val_loss: 0.8655
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6979 - loss: 0.9102 - val_accuracy: 0.7917 - val_loss: 0.8404
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7188 - loss: 0.8809 - val_accuracy: 0.7917 - val_loss: 0.8163
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7500 - loss: 0.8547 - val_accuracy: 0.7500 - val_loss: 0.7924
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7604 - loss: 0.8276 - val_accuracy: 0.7500 - val_loss:

In [10]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test_encoded,
    verbose=0
)

print("Test Categorical Cross-Entropy:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy, 4))

Test Categorical Cross-Entropy: 0.134
Test Accuracy: 0.9667


In [11]:
# Predicted probabilities
y_prob = model.predict(X_test, verbose=0)

# Convert probabilities to class indices
y_pred = np.argmax(y_prob, axis=1)

print("Actual labels:    ", y_test[:10])
print("Predicted labels: ", y_pred[:10])
print("Probabilities:\n", np.round(y_prob[:5], 3))

Actual labels:     [0 2 1 1 0 1 0 0 2 1]
Predicted labels:  [0 2 1 1 0 1 0 0 2 1]
Probabilities:
 [[0.999 0.001 0.001]
 [0.017 0.146 0.837]
 [0.103 0.884 0.014]
 [0.034 0.959 0.006]
 [0.999 0.    0.   ]]
